# Week 13: Mini Project v3.1 --- Improvement & Documentation
### PHASE 6: Proving Mastery

*📚 Object Oriented Programming · ⏱️ 3 Hours · 👨‍🏫 Dr. Arif Solmaz*

---

## 🎯 Learning Objectives

By the end of this week, you will be able to:

| # | Objective |
|---|----------|
| 1 | Write clear docstrings for classes and methods |
| 2 | Add basic type hints to function signatures |
| 3 | Refactor code to simplify the public API |
| 4 | Use a code review checklist to evaluate quality |
| 5 | Improve the Mechatronics Toolkit from Week 12 |

---

## Overview

This week we take the **Mechatronics Toolkit** you built in Week 12 and make it *better*.

We are **not** adding new features. Instead, we focus on:

- **Documentation** --- so others can understand your code
- **Type hints** --- so readers know what types go in and come out
- **Refactoring** --- so the API is simple and clean
- **Code review** --- so you can check quality yourself

> *"Code is read much more often than it is written."* --- Guido van Rossum

## 🎯 Core Mastery Connection

**Core Mastery: "I can model a system as collaborating objects."**

Good components have clear APIs. When objects collaborate through composition, the interfaces between them — the method signatures, the docstrings, the type hints — determine how easily you can swap, extend, or debug each part. This week you refactor, document, and simplify the interfaces between your composed objects so the system becomes truly professional and maintainable.

---

# Part 1: Review Your Week 12 Code

Before improving anything, let us **review** what we built last week.

Your Week 12 Mechatronics Toolkit should have these classes:

| Class | Purpose |
|-------|--------|
| `Sensor` | Reads or simulates sensor values |
| `Filter` | Cleans noisy data (moving average or median) |
| `Logger` | Saves data to a CSV file |

**Figure 13.1** --- Week 12 toolkit structure

```
Sensor  --->  Filter  --->  Logger
(read data)   (clean)       (save)
```

Let us start by recreating a **basic version** of the toolkit so everyone has the same starting point.

In [ ]:
# Week 12 Baseline Code --- Starting Point
# This is a simplified version of the Mechatronics Toolkit

import random

class Sensor:
    """A simulated sensor that generates noisy readings."""

    def __init__(self, name, base_value, noise_level):
        self.name = name
        self.base_value = base_value
        self.noise_level = noise_level
        self.readings = []

    def read(self):
        noise = random.uniform(-self.noise_level, self.noise_level)
        value = self.base_value + noise
        self.readings.append(value)
        return value

    def read_many(self, count):
        return [self.read() for _ in range(count)]

    def __repr__(self):
        return f"Sensor('{self.name}', base={self.base_value})"


class Filter:
    """Filters noisy data using a moving average."""

    def __init__(self, window_size):
        self.window_size = window_size

    def apply(self, data):
        result = []
        for i in range(len(data)):
            start = max(0, i - self.window_size + 1)
            window = data[start:i + 1]
            avg = sum(window) / len(window)
            result.append(avg)
        return result

    def __repr__(self):
        return f"Filter(window={self.window_size})"


class Logger:
    """Logs data to a CSV file."""

    def __init__(self, filename):
        self.filename = filename

    def save(self, data, header):
        with open(self.filename, 'w') as f:
            f.write(header + '\n')
            for i, value in enumerate(data):
                f.write(f"{i},{value:.4f}\n")

    def __repr__(self):
        return f"Logger('{self.filename}')"


# Quick test
sensor = Sensor("temperature", 25.0, 2.0)
print(sensor)
print("3 readings:", sensor.read_many(3))

---

# Part 2: Adding Docstrings

A **docstring** is a string that describes what a class or method does.

Python uses triple quotes for docstrings: `"""..."""`

### Why Docstrings?

| Without docstrings | With docstrings |
|---|---|
| You must read the code to understand it | You read the description |
| `help(Sensor)` shows nothing useful | `help(Sensor)` shows clear docs |
| Other developers get confused | Other developers say thank you |

### Docstring Format

We will use a simple format:

```python
def method_name(self, param1, param2):
    """Short description of what this method does.

    Args:
        param1: What param1 is.
        param2: What param2 is.

    Returns:
        What the method returns.
    """
```

**Figure 13.2** --- Docstring structure

```
+---------------------------+
| Short description         |  <-- First line: one sentence
|                           |
| Args:                     |  <-- Parameters explained
|     param1: ...           |
|     param2: ...           |
|                           |
| Returns:                  |  <-- What comes back
|     ...                   |
+---------------------------+
```


> 💡 **Tip:** Good docstrings answer three questions: *What does it do?* *What does it take?* *What does it return?*

| Docstring Style | Used By | Example |
|---|---|---|
| **Google style** | Google, this course | `Args:`, `Returns:` sections |
| **NumPy style** | NumPy, SciPy, pandas | `Parameters`, `Returns` sections |
| **Sphinx style** | Sphinx documentation | `:param name:`, `:returns:` |


In [ ]:
# Example: Sensor class WITH proper docstrings

class Sensor:
    """A simulated sensor that generates noisy readings.

    This class simulates a real sensor by adding random noise
    to a base value. Useful for testing signal processing code.

    Args:
        name: Name of the sensor (e.g., 'temperature').
        base_value: The true value the sensor measures.
        noise_level: Maximum noise added to readings.
    """

    def __init__(self, name, base_value, noise_level):
        """Create a new Sensor.

        Args:
            name: Sensor name for identification.
            base_value: The center value of readings.
            noise_level: Half-width of uniform noise.
        """
        self.name = name
        self.base_value = base_value
        self.noise_level = noise_level
        self.readings = []

    def read(self):
        """Take a single sensor reading.

        Returns:
            A float value: base_value + random noise.
        """
        noise = random.uniform(-self.noise_level, self.noise_level)
        value = self.base_value + noise
        self.readings.append(value)
        return value

    def read_many(self, count):
        """Take multiple sensor readings.

        Args:
            count: Number of readings to take.

        Returns:
            A list of float values.
        """
        return [self.read() for _ in range(count)]

    def __repr__(self):
        """Return a string representation of the sensor."""
        return f"Sensor('{self.name}', base={self.base_value})"


# Now help() gives useful output!
help(Sensor)

In [ ]:
# You can also access docstrings directly

print("Class docstring:")
print(Sensor.__doc__)

print("\nMethod docstring:")
print(Sensor.read.__doc__)

---

# Part 3: Basic Type Hints

**Type hints** tell the reader what type each parameter and return value should be.

They do **not** change how the code runs. They are just **labels** for humans.

### Before and After

| Without type hints | With type hints |
|---|---|
| `def read_many(self, count):` | `def read_many(self, count: int) -> list[float]:` |
| What is `count`? A string? A float? | `count` is an `int`, returns `list[float]` |

### Common Type Hints

| Type | Example | Meaning |
|------|---------|--------|
| `int` | `count: int` | Integer number |
| `float` | `value: float` | Decimal number |
| `str` | `name: str` | Text string |
| `bool` | `active: bool` | True or False |
| `list[float]` | `data: list[float]` | List of floats |
| `None` | `-> None` | Returns nothing |

**Figure 13.1** — Sensor class with type hints on all parameters and returns

In [ ]:
# Example: Sensor with type hints

class Sensor:
    """A simulated sensor that generates noisy readings."""

    def __init__(self, name: str, base_value: float, noise_level: float) -> None:
        """Create a new Sensor.

        Args:
            name: Sensor name for identification.
            base_value: The center value of readings.
            noise_level: Half-width of uniform noise.
        """
        self.name: str = name
        self.base_value: float = base_value
        self.noise_level: float = noise_level
        self.readings: list[float] = []

    def read(self) -> float:
        """Take a single sensor reading.

        Returns:
            A float value: base_value + random noise.
        """
        noise = random.uniform(-self.noise_level, self.noise_level)
        value = self.base_value + noise
        self.readings.append(value)
        return value

    def read_many(self, count: int) -> list[float]:
        """Take multiple sensor readings.

        Args:
            count: Number of readings to take.

        Returns:
            A list of float values.
        """
        return [self.read() for _ in range(count)]

    def __repr__(self) -> str:
        """Return a string representation of the sensor."""
        return f"Sensor('{self.name}', base={self.base_value})"


# Test it --- works exactly the same
s = Sensor("pressure", 101.3, 0.5)
print(s)
print("Reading:", s.read())

**Figure 13.2** — Filter class with type hints and docstrings

In [ ]:
# Example: Filter with type hints and docstrings

class Filter:
    """Filters noisy data using a moving average.

    A simple digital filter that smooths sensor data
    by averaging values in a sliding window.
    """

    def __init__(self, window_size: int) -> None:
        """Create a new Filter.

        Args:
            window_size: Number of values to average.
        """
        self.window_size: int = window_size

    def apply(self, data: list[float]) -> list[float]:
        """Apply the moving average filter to data.

        Args:
            data: List of raw sensor values.

        Returns:
            List of filtered (smoothed) values.
        """
        result: list[float] = []
        for i in range(len(data)):
            start = max(0, i - self.window_size + 1)
            window = data[start:i + 1]
            avg = sum(window) / len(window)
            result.append(avg)
        return result

    def __repr__(self) -> str:
        """Return a string representation of the filter."""
        return f"Filter(window={self.window_size})"


# Test
f = Filter(3)
raw = [10.0, 12.0, 8.0, 11.0, 9.0]
print("Raw:     ", raw)
print("Filtered:", f.apply(raw))

---

# Part 4: Refactoring --- Simplify the API

**Refactoring** means changing the code structure without changing what it does.

Our goal: make the toolkit **easier to use**.

### What is an API?

API = **Application Programming Interface** = the methods other people call.

A good API is:
- **Simple** --- few methods to learn
- **Clear** --- method names explain what they do
- **Consistent** --- similar things work in similar ways

**Figure 13.3** --- Good vs. bad API

```
BAD API:                          GOOD API:
sensor.getData()                  sensor.read()
sensor.getSingleData()            sensor.read_many(10)
sensor.getMultipleData(10)
sensor.fetchReading()

4 methods, confusing!             2 methods, clear!
```

### Common Refactoring Steps

| Problem | Solution |
|---------|----------|
| Too many parameters | Use default values |
| Confusing method names | Rename to be clear |
| Repeated code | Extract into a helper method |
| Long method | Break into smaller methods |

In [ ]:
# Refactoring Example: Logger with default values

# BEFORE: user must provide everything
class LoggerOld:
    def __init__(self, filename):
        self.filename = filename

    def save(self, data, header):
        with open(self.filename, 'w') as f:
            f.write(header + '\n')
            for i, value in enumerate(data):
                f.write(f"{i},{value:.4f}\n")


# AFTER: sensible defaults, clearer API
class Logger:
    """Logs sensor data to a CSV file.

    Provides simple methods to save and display data.
    """

    def __init__(self, filename: str = "data.csv") -> None:
        """Create a new Logger.

        Args:
            filename: Output file path. Default is 'data.csv'.
        """
        self.filename: str = filename

    def save(self, data: list[float], header: str = "index,value") -> None:
        """Save data to a CSV file.

        Args:
            data: List of values to save.
            header: CSV header row. Default is 'index,value'.
        """
        with open(self.filename, 'w') as f:
            f.write(header + '\n')
            for i, value in enumerate(data):
                f.write(f"{i},{value:.4f}\n")
        print(f"Saved {len(data)} values to '{self.filename}'")

    def preview(self, data: list[float], n: int = 5) -> None:
        """Print the first n values of the data.

        Args:
            data: List of values to preview.
            n: Number of values to show. Default is 5.
        """
        print(f"Preview (first {n} of {len(data)} values):")
        for i, value in enumerate(data[:n]):
            print(f"  [{i}] {value:.4f}")

    def __repr__(self) -> str:
        """Return a string representation of the logger."""
        return f"Logger('{self.filename}')"


# BEFORE: user had to specify everything
# old = LoggerOld("data.csv")
# old.save([1.0, 2.0], "index,value")  # must provide header

# AFTER: simple and clean
logger = Logger()  # default filename
logger.preview([1.1, 2.2, 3.3, 4.4, 5.5, 6.6])
logger.save([1.1, 2.2, 3.3])  # default header

**Figure 13.3** — Refactoring: Pipeline convenience class for one-line usage

In [ ]:
# Refactoring Example: Adding a convenience method

# Instead of making the user do this:
#   sensor = Sensor("temp", 25.0, 2.0)
#   raw = sensor.read_many(20)
#   filt = Filter(5)
#   clean = filt.apply(raw)
#   log = Logger("output.csv")
#   log.save(clean)

# We can add a Pipeline class:

class Pipeline:
    """Connects a Sensor, Filter, and Logger together.

    A convenience class that runs the full data pipeline
    in a single method call.
    """

    def __init__(self, sensor: Sensor, filt: Filter, logger: Logger) -> None:
        """Create a new Pipeline.

        Args:
            sensor: The sensor to read data from.
            filt: The filter to clean the data.
            logger: The logger to save results.
        """
        self.sensor = sensor
        self.filt = filt
        self.logger = logger

    def run(self, count: int = 20) -> list[float]:
        """Run the full pipeline: read, filter, save.

        Args:
            count: Number of readings to take.

        Returns:
            The filtered data.
        """
        # Step 1: Read
        raw = self.sensor.read_many(count)
        print(f"Read {count} values from {self.sensor.name}")

        # Step 2: Filter
        clean = self.filt.apply(raw)
        print(f"Filtered with {self.filt}")

        # Step 3: Save
        self.logger.save(clean)

        return clean

    def __repr__(self) -> str:
        """Return a string representation of the pipeline."""
        return f"Pipeline({self.sensor} -> {self.filt} -> {self.logger})"


# Simple one-line usage!
pipe = Pipeline(
    sensor=Sensor("temperature", 25.0, 2.0),
    filt=Filter(5),
    logger=Logger("temp_data.csv")
)
print(pipe)
result = pipe.run(10)

---

# Part 5: Code Review Checklist

A **code review** is when you (or someone else) checks code for quality.

Use this checklist to review your own code:

### Code Review Checklist

| # | Check | Question to Ask |
|---|-------|----------------|
| 1 | **Naming** | Are class and method names clear? |
| 2 | **Docstrings** | Does every class and public method have a docstring? |
| 3 | **Type hints** | Do methods have type hints for parameters and returns? |
| 4 | **Defaults** | Do parameters have sensible default values where possible? |
| 5 | **Single job** | Does each class do one thing? (SRP) |
| 6 | **Error handling** | Does the code handle bad input? |
| 7 | **Repr** | Does every class have a `__repr__`? |
| 8 | **No magic numbers** | Are constants named, not hardcoded? |

**Figure 13.4** --- Code review process

```
Write Code --> Review with Checklist --> Fix Issues --> Review Again
    ^                                                      |
    |______________________________________________________|    
```

In [ ]:
# Example: Code that FAILS the review checklist

class S:  # Bad name!
    def __init__(self, n, b, nl):
        # No docstring, no type hints, bad parameter names
        self.n = n
        self.b = b
        self.nl = nl
        self.r = []

    def g(self):  # What does 'g' mean?
        import random
        v = self.b + random.uniform(-self.nl, self.nl)
        self.r.append(v)
        return v


# Can you read this? Neither can anyone else!
x = S("temp", 25, 2)
print(x.g())  # What does this do?

**Figure 13.4** — Code after applying the review checklist

In [ ]:
# Same code AFTER applying the checklist

class TemperatureSensor:  # Clear name
    """A simulated temperature sensor."""  # Docstring

    def __init__(self, name: str, base_temp: float, noise: float = 1.0) -> None:
        """Create a temperature sensor.

        Args:
            name: Sensor identifier.
            base_temp: Expected temperature in Celsius.
            noise: Maximum noise range. Default is 1.0.
        """
        self.name = name
        self.base_temp = base_temp
        self.noise = noise
        self.readings: list[float] = []

    def read(self) -> float:
        """Take a single temperature reading.

        Returns:
            Temperature in Celsius with noise.
        """
        value = self.base_temp + random.uniform(-self.noise, self.noise)
        self.readings.append(value)
        return value

    def __repr__(self) -> str:
        """Return a readable string."""
        return f"TemperatureSensor('{self.name}', {self.base_temp}C)"


# Now it is clear and readable!
sensor = TemperatureSensor("room", 22.5)
print(sensor)
print(f"Reading: {sensor.read():.1f}C")

---

# Part 6: Improved Toolkit

Now let us put everything together into an **improved** Mechatronics Toolkit.

This version includes:
- Docstrings on every class and method
- Type hints on all parameters and returns
- Default values for convenience
- A Pipeline class for easy usage
- Input validation

**Figure 13.5** --- Improved toolkit architecture

```
+------------------+     +------------------+     +------------------+
|     Sensor       |     |     Filter       |     |     Logger       |
|------------------|     |------------------|     |------------------|
| + name: str      |     | + window: int    |     | + filename: str  |
| + base: float    |     |                  |     |                  |
| + noise: float   |     | + apply(data)    |     | + save(data)     |
| + readings: list |     |   -> list[float] |     | + preview(data)  |
|                  |     |                  |     |                  |
| + read() -> float|     +------------------+     +------------------+
| + read_many(n)   |
|   -> list[float] |
+------------------+
           \
            \        +------------------+
             +------>|    Pipeline      |
                     |------------------|
                     | + sensor         |
                     | + filter         |
                     | + logger         |
                     |                  |
                     | + run(count)     |
                     |   -> list[float] |
                     +------------------+
```

---

# Exercises

> **Composition lens:** Every exercise below improves how your components communicate. Clear docstrings tell other developers what a component expects and returns. Type hints define the contract between composed objects. When you refactor an API, you are making it easier to plug components together — the essence of composition.

Complete the following exercises to practice documentation, type hints, and refactoring.

---

# Exercises

Complete the following exercises to practice documentation, type hints, and refactoring.

In [ ]:
# ✏️ [EX1] Add Docstrings to Actuator
#
# The class below has NO docstrings.
# Add a class docstring and method docstrings (Args + Returns).
#
# Hint: Follow the format shown in Part 2.

class Actuator:
    def __init__(self, name, max_power):
        self.name = name
        self.max_power = max_power
        self.current_power = 0.0

    def set_power(self, power):
        if power < 0:
            power = 0
        if power > self.max_power:
            power = self.max_power
        self.current_power = power
        return self.current_power

    def stop(self):
        self.current_power = 0.0

    def __repr__(self):
        return f"Actuator('{self.name}', power={self.current_power}/{self.max_power})"


# Test your changes
a = Actuator("motor", 100.0)
a.set_power(75)
print(a)
help(Actuator)

### Exercise 2: Add Type Hints to Actuator (Easy)

Take your Actuator class from EX1 and add type hints to ALL methods (parameters and return types). stop takes nothing and returns None.

<details>
<summary>💡 Hint</summary>
Common types: <code>int</code>, <code>float</code>, <code>str</code>, <code>bool</code>, <code>list[float]</code>. Use <code>-> None</code> for methods that return nothing.
</details>

In [ ]:
# ✏️ [EX2] Add Type Hints to Actuator
#
# Take your Actuator class from EX1 and add type hints
# to ALL methods (parameters and return types).
#
# Hint: set_power takes a float and returns a float.
#       stop takes nothing and returns None.

# YOUR CODE HERE


### Exercise 3: Refactor: Add Default Values (Easy)

Refactor the Sensor class so that: - noise_level has a default of 1.0 - read_many has a default count of 10

<details>
<summary>💡 Hint</summary>
Use <code>def __init__(self, name: str, noise_level: float = 1.0)</code> syntax for defaults.
</details>

In [ ]:
# ✏️ [EX3] Refactor: Add Default Values
#
# Refactor the Sensor class so that:
# - noise_level has a default of 1.0
# - read_many has a default count of 10
#
# Then create a sensor using ONLY: Sensor("voltage", 5.0)
#
# Hint: def __init__(self, name: str, base_value: float, noise_level: float = 1.0)

# YOUR CODE HERE


### Exercise 4: Write a Docstring for a Method (Easy)

Write a proper docstring for this method. Include: short description, Args, and Returns.

<details>
<summary>💡 Hint</summary>
Read what the code does first, then describe it. Include the parameter types and what it returns.
</details>

In [ ]:
# ✏️ [EX4] Write a Docstring for a Method
#
# Write a proper docstring for this method.
# Include: short description, Args, and Returns.
#
# Hint: Look at what the code actually does.

class Statistics:
    def compute_average(self, values):
        # YOUR DOCSTRING HERE
        if not values:
            return 0.0
        return sum(values) / len(values)

    def compute_range(self, values):
        # YOUR DOCSTRING HERE
        if not values:
            return 0.0
        return max(values) - min(values)


# Test
stats = Statistics()
data = [10, 20, 30, 40, 50]
print(f"Average: {stats.compute_average(data)}")
print(f"Range: {stats.compute_range(data)}")

### Exercise 5: Code Review: Find the Problems (Medium)

Review the class below using the checklist from Part 5. List at least 4 problems in a comment, then fix them.

<details>
<summary>💡 Hint</summary>
Check: naming conventions, missing docstrings, no type hints, magic numbers, missing validation, unclear variable names.
</details>

In [ ]:
# ✏️ [EX5] Code Review: Find the Problems
#
# Review the class below using the checklist from Part 5.
# List at least 4 problems in a comment, then fix them.
#
# Hint: Check naming, docstrings, type hints, defaults, repr.

class D:
    def __init__(self, d, t):
        self.d = d
        self.t = t

    def c(self):
        return self.d / self.t


# Problems found:
# 1. ...
# 2. ...
# 3. ...
# 4. ...

# YOUR FIXED VERSION HERE


### Exercise 6: Add Input Validation (Medium)

Add input validation to the Filter class: - window_size must be at least 1 (raise ValueError if not) - data passed to apply() must not be empty (raise ValueError)

<details>
<summary>💡 Hint</summary>
Use <code>if value < 0: raise ValueError('...')</code> pattern. Validate in <code>__init__</code> and in methods.
</details>

In [ ]:
# ✏️ [EX6] Add Input Validation
#
# Add input validation to the Filter class:
# - window_size must be at least 1 (raise ValueError if not)
# - data passed to apply() must not be empty (raise ValueError)
#
# Include docstrings and type hints.
#
# Hint: Use 'if window_size < 1: raise ValueError("...")'

# YOUR CODE HERE


### Exercise 7: Create a MedianFilter Class (Medium)

Create a MedianFilter class with: - Proper docstrings - Type hints

<details>
<summary>💡 Hint</summary>
Median = middle value of sorted list. Use <code>sorted(window)[len(window)//2]</code>.
</details>

In [ ]:
# ✏️ [EX7] Create a MedianFilter Class
#
# Create a MedianFilter class with:
# - Proper docstrings
# - Type hints
# - A window_size parameter with default of 3
# - An apply() method that uses the median instead of average
# - A __repr__ method
#
# Hint: To find the median of a list, sort it and take the middle value.
#       sorted_window = sorted(window)
#       median = sorted_window[len(sorted_window) // 2]

# YOUR CODE HERE


### Exercise 8: Refactor the Pipeline (Medium)

Add a 'summary' method to the Pipeline class that prints: - Sensor name and base value - Filter window size

<details>
<summary>💡 Hint</summary>
The summary method should access attributes from each component: <code>self._sensor.name</code>, <code>self._filter.window_size</code>.
</details>

In [ ]:
# ✏️ [EX8] Refactor the Pipeline
#
# Add a 'summary' method to the Pipeline class that prints:
# - Sensor name and base value
# - Filter window size
# - Logger filename
# - Number of readings taken so far
#
# Include docstring and type hints.
#
# Hint: Use self.sensor.name, len(self.sensor.readings), etc.

# YOUR CODE HERE


### Exercise 9: Complete Toolkit with Documentation (Challenge)

Create a complete mini toolkit with these classes: - MotorController: controls a motor (set_speed, stop, get_status) - SpeedSensor: reads motor speed (read, read_many)

<details>
<summary>💡 Hint</summary>
Start with the MotorController class. Add full docstrings and type hints. Then build SpeedSensor and Logger that work together.
</details>

In [ ]:
# ✏️ [EX9] Complete Toolkit with Documentation
#
# Create a complete mini toolkit with these classes:
# - MotorController: controls a motor (set_speed, stop, get_status)
# - SpeedSensor: reads motor speed (read, read_many)
#
# Requirements:
# - Every class and method must have a docstring
# - Every method must have type hints
# - Use default values where appropriate
# - Include __repr__ for both classes
# - Add input validation (speed must be 0-100)
#
# Hint: Start with __init__, then add methods one by one.

# YOUR CODE HERE


### Exercise 10: Code Review Report (Challenge)

Review your OWN code from EX9 using the checklist. Print a report showing pass/fail for each check. Code Review Report

<details>
<summary>💡 Hint</summary>
Check each item programmatically where possible: <code>hasattr(obj, '__doc__')</code> checks for docstrings.
</details>

In [ ]:
# ✏️ [EX10] Code Review Report
#
# Review your OWN code from EX9 using the checklist.
# Print a report showing pass/fail for each check.
#
# Expected output format:
# Code Review Report
# ==================
# [PASS] Naming: class and method names are clear
# [PASS] Docstrings: all classes and methods documented
# ...
#
# Hint: Just use print statements. No automation needed.

# YOUR CODE HERE


---

## 🌉 Bridge to Next Week

This week we focused on **quality** — adding docstrings, type hints, input validation, and using a code review checklist to make our code professional and maintainable.

**Next week** is the final week. You will prepare and deliver your **final architecture presentation**. This includes:

- A **demo notebook** that showcases your Mechatronics Toolkit in action
- A **test report** demonstrating that your classes and methods pass their unit tests
- A short **presentation** walking through your design decisions, class hierarchy, and how the pieces fit together

Start thinking about how you would explain your toolkit to someone who has never seen it. What are the key classes? How do they interact? What design patterns did you use?

---
## 📮 Submission

Follow the two steps below to submit your work.

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 1: Fill in your info below, then run this cell#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━STUDENT_ID    = ""     # e.g. "2024001234"STUDENT_NAME  = ""     # e.g. "Ahmet Yılmaz"STUDENT_EMAIL = ""     # e.g. "ahmet.yilmaz@istun.edu.tr"CLASS_CODE    = ""     # code given in class#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# Don't change anything below this line#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import re as _re_errors = []if not _re.match(r"^\d{6,12}$", STUDENT_ID):    _errors.append("❌ Student ID must be 6-12 digits")if len(STUDENT_NAME.strip().split()) < 2:    _errors.append("❌ Enter first and last name")if not STUDENT_EMAIL.strip().lower().endswith("@istun.edu.tr") or len(STUDENT_EMAIL.strip()) < 16:    _errors.append("❌ Use your @istun.edu.tr email")if len(CLASS_CODE.strip()) < 4:    _errors.append("❌ Invalid class code")if _errors:    for _e in _errors:        print(_e)    print("\n⚠️  Fix the errors above and run this cell again.")else:    print(f"✅ Info OK — {STUDENT_NAME} ({STUDENT_ID})")    print(f"   {STUDENT_EMAIL}")    print(f"\n👉 Now run the NEXT cell to submit.")

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 2: Run this cell to submit#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import json, re, os, urllib.requestWEEK = "Week_13"URL  = "https://script.google.com/macros/s/AKfycbyf1D3HGSAX4MoIhNlAuWlGrFyyvbM5MIv7ZsLxrVDlATUihrRGEAaibvIZYlCfd8Me/exec"try:    _sid = STUDENT_ID.strip()    _sname = STUDENT_NAME.strip()    _semail = STUDENT_EMAIL.strip().lower()    _scode = CLASS_CODE.strip().upper()except NameError:    raise SystemExit("❌ Run the cell above first to set your info!")if not _sid or not _sname or not _semail or not _scode:    raise SystemExit("❌ Run the cell above first — some fields are empty.")_answers = {}try:    _ipy = get_ipython()    _hist = _ipy.history_manager.get_range(output=False)    for _sess, _line, _src in _hist:        _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)        if _m:            _ex_id = "ex" + _m.group(1)            _lines = _src.split("\n")            _clean = "\n".join(_lines[1:]).strip()            _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}except Exception:    passif not _answers:    try:        for _src in In:            if not _src: continue            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}    except NameError:        passif not _answers:    _nb_path = None    try:        _nb_path = __vsc_ipynb_file__    except NameError:        _candidates = [f for f in os.listdir(".") if f.endswith(".ipynb") and WEEK in f]        if len(_candidates) == 1: _nb_path = _candidates[0]    if _nb_path and os.path.exists(str(_nb_path)):        with open(str(_nb_path), "r", encoding="utf-8") as _f:            _nb = json.load(_f)        for _cell in _nb["cells"]:            if _cell["cell_type"] != "code": continue            _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}print(f"📝 Found {len(_answers)} exercise(s): {', '.join(sorted(_answers.keys()))}")if not _answers:    print("\n⚠️  No exercise answers found!")    print("Make sure you RAN all exercise cells before submitting.")    raise SystemExit()_data = json.dumps({"week": WEEK, "studentId": _sid, "studentName": _sname, "studentEmail": _semail, "classCode": _scode, "source": "oop-notebook", "timeOnPage": 0, "answers": _answers}).encode("utf-8")print("📡 Submitting...")try:    _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")    _resp = urllib.request.urlopen(_req, timeout=30)    _result = json.loads(_resp.read().decode())    if _result.get("success"):        print(f"\n✅ {_result['message']}")        print("📧 Check your email for confirmation.")    else:        print(f"\n❌ {_result.get('message', 'Submission failed')}")except Exception as _e:    try:        _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")        urllib.request.urlopen(_req, timeout=10)    except: pass    print(f"\n⚠️  Request sent — check your email for confirmation.")    print(f"(If no email arrives, try again or contact your instructor)")